# Storm: SV + TR sidecar + Hail `annotate_svs`

This notebook exercises:

1. **`storm-tr-sidecar`** → `sites.parquet` (optional; or point to an existing sidecar).
2. **`hl.import_vcf`** → **`storm.annotate_svs`** → MatrixTable with `allele_id`, `dosage`, and optional **`tr`** row struct.

**Setup**

- Install Hail **0.2.134** (matches Terra); see [Hail install docs](https://hail.is/docs/0.2/getting_started.html) (Java 11 on Mac ARM64, Python 3.10+). Example: `pip install 'storm[hail]'` from this repo pins that version.
- Install `storm` in this env (`pip install -e ./python` or `maturin develop` from the repo).

Start Jupyter from the **repository root** (`storm/`), or adjust `REPO_ROOT` below.

In [1]:
from pathlib import Path
import sys

# Repository root: directory that contains python/storm/
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python" / "storm").is_dir():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "python") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "python"))

print("REPO_ROOT =", REPO_ROOT.resolve())

REPO_ROOT = /Users/kiran/repositories/storm


In [29]:
import importlib
import storm
import storm.saige_prep
importlib.reload(storm.saige_prep)
importlib.reload(storm)

print("storm version:", storm.version())

storm version: 0.0.0


In [3]:
import os
import subprocess
import hail as hl

# Force Java 11 for this kernel session before Hail init.
def _java11_home() -> str | None:
    try:
        p = subprocess.run(["/usr/libexec/java_home", "-v", "11"], capture_output=True, text=True)
        jh = p.stdout.strip()
        return jh if jh else None
    except Exception:
        return None

_java11 = _java11_home()
if _java11:
    os.environ["JAVA_HOME"] = _java11
    os.environ["PATH"] = f"{_java11}/bin:" + os.environ.get("PATH", "")


def _java_version_text() -> str:
    try:
        p = subprocess.run(["java", "-version"], capture_output=True, text=True)
        return (p.stderr or p.stdout).strip().splitlines()[0]
    except Exception:
        return "java -version unavailable"

JAVA_VERSION = _java_version_text()
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("Java:", JAVA_VERSION)

# Local backend is fine for smoke tests; use cluster Spark/Dataproc for full cohorts.
hl.init_local(quiet=True, default_reference="GRCh38")
print("Hail backend:", hl.current_backend())
print("Hail default reference:", hl.default_reference())

if "version \"11" not in JAVA_VERSION:
    raise RuntimeError(
        "Java 11 is required for reliable Hail 0.2.x local use. "
        "Restart kernel and ensure /usr/libexec/java_home -v 11 is available."
    )

Loading BokehJS ...

JAVA_HOME: /Library/Java/JavaVirtualMachines/temurin-11.jdk/Contents/Home
Java: openjdk version "11.0.30" 2026-01-20
Hail backend: <hail.backend.local_backend.LocalBackend object at 0x341b00cd0>
Hail default reference: GRCh38


## Paths

Set your union SV VCF/BCF (and optional existing `sites.parquet`). For a quick smoke test, use a small subset.

In [18]:
CATALOG = REPO_ROOT / "data" / "tr_loci_hg38" / "tr_loci_hg38.tsv"

VCF_PATH = REPO_ROOT / "union.chr22_25000000_35000000.vcf.bgz"
SIDECAR_DIR = REPO_ROOT / "sidecar_out_notebook"
SITES_PARQUET = SIDECAR_DIR / "sites.parquet"

print("VCF exists:", VCF_PATH.exists(), VCF_PATH)
print("Catalog exists:", CATALOG.exists(), CATALOG)

VCF exists: True /Users/kiran/repositories/storm/union.chr22_25000000_35000000.vcf.bgz
Catalog exists: True /Users/kiran/repositories/storm/data/tr_loci_hg38/tr_loci_hg38.tsv


## Build TR site sidecar

Skip if you already have `sites.parquet`. For large cohorts, do **not** use `--with-entries`.

In [5]:
r = storm.build_tr_sidecar(VCF_PATH, CATALOG, SIDECAR_DIR, with_entries=False)
print("Wrote", r.sites_path, "variants:", r.n_variants)

Wrote /Users/kiran/repositories/storm/sidecar_out_notebook/sites.parquet variants: 18354


## Import VCF + `storm.annotate_svs`

Join key: sidecar **`variant_id`** = MatrixTable **`allele_id`** after split (not the raw multi-allelic `ID` string unless it is one comma-separated token per ALT).

In [6]:
mt0 = hl.import_vcf(str(VCF_PATH), force_bgz=True, reference_genome="GRCh38")
mt1 = storm.annotate_svs(mt0, tr_sidecar_sites=str(SITES_PARQUET))

In [7]:
print("Rows:", mt1.count_rows(), "Cols:", mt1.count_cols())
preview = ["allele_id", "id_parse_ok", "svlen", "tr"]
mt1.rows().select(*preview).show(5)

Rows: 18354 Cols: 12680


+----------------+
| locus          |
+----------------+
| locus<GRCh38>  |
+----------------+
| chr22:25000196 |
| chr22:25000734 |
| chr22:25001794 |
| chr22:25003400 |
| chr22:25003400 |
+----------------+

+------------------------------------------------------------------------------+
| alleles                                                                      |
+------------------------------------------------------------------------------+
| array<str>                                                                   |
+------------------------------------------------------------------------------+
| ["TTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGCGGGATCTCGGCT... |
| ["CTTGAACTCCTGGCCTCAAGTGAATCCGCCTGCCTCGGCCTCCGGAAG","C"]                     |
| ["A","ACGTGCCTGTTGGTCTTTCAGCTACAGAT"]                                        |
| ["A","ATTTTTTTTTTTTTTTTTTTT"]                                                |
| ["A","ATTTTTTTTTTTTTTTTTTTTTT"]                                              |
+------------------------------------------------------------------------------+

+--------------------------+-------------+-------+-------------------+
| allele_id                | id_parse_ok | svlen | tr.schema_version |
+--------------------------+-------------+-------+-------------------+
| str                      |        bool | int32 | str               |
+--------------------------+-------------+-------+-------------------+
| "chr22-25000197-DEL-296" |        True |   296 | "1"               |
| "chr22-25000735-DEL-47"  |        True |    47 | "1"               |
| "chr22-25001795-INS-28"  |        True |    28 | "1"               |
| "chr22-25003401-INS-20"  |        True |    20 | "1"               |
| "chr22-25003401-INS-22"  |        True |    22 | "1"               |
+--------------------------+-------------+-------+-------------------+

+--------------------+-----------+--------------+------------+
| tr.ruleset_version | tr.contig | tr.pos_start | tr.pos_end |
+--------------------+-----------+--------------+------------+
| str                | str       |        int32 |      int32 |
+--------------------+-----------+--------------+------------+
| "1"                | "chr22"   |     25000195 |   25000492 |
| "1"                | "chr22"   |     25000733 |   25000781 |
| "1"                | "chr22"   |     25001793 |   25001822 |
| "1"                | "chr22"   |     25003399 |   25003420 |
| "1"                | "chr22"   |     25003399 |   25003422 |
+--------------------+-----------+--------------+------------+

+------------------------------------------------------------------------------+
| tr.ref                                                                       |
+------------------------------------------------------------------------------+
| str                                                                          |
+------------------------------------------------------------------------------+
| "TTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGCGGGATCTCGGCTC... |
| "CTTGAACTCCTGGCCTCAAGTGAATCCGCCTGCCTCGGCCTCCGGAAG"                           |
| "A"                                                                          |
| "A"                                                                          |
| "A"                                                                          |
+------------------------------------------------------------------------------+

+---------------------------------+-----------+----------+--------+----------+
| tr.alt                          | tr.svtype | tr.svlen | tr.end | tr.motif |
+---------------------------------+-----------+----------+--------+----------+
| str                             | str       |    int32 |  int32 | str      |
+---------------------------------+-----------+----------+--------+----------+
| "T"                             | "DEL"     |      296 |     NA | ""       |
| "C"                             | "DEL"     |       47

### Rows with `repeat_units_estimate`

After `annotate_svs` with a TR sidecar, estimates live in `tr.repeat_units_estimate`. Filter with `hl.is_defined` (and optionally `hl.is_finite` to drop NaN).

**Display:** Inspect with `mt_ru.rows().select(...).show()` — not `mt_ru.show()`. On a MatrixTable, `show()` is the sample×variant grid; Hail may print “first **0** of *N* **columns**” (*N* = samples) when it is not drawing genotype columns.

**Data:** The sidecar sets `repeat_units_estimate` only when `INFO/SVLEN` and `INFO/MOTIF_SIZE` exist (multi-value `SVLEN` uses the first allele’s length). Rebuild `sites.parquet` after upgrading `storm` if you had empty estimates before.

In [8]:
# Requires mt1 from the previous cell and a sidecar (so row field `tr` exists).
mt_ru = mt1.filter_rows(
    hl.is_defined(mt1.tr.rule_applicable)
)

print("Rows with defined rule_applicable:", mt_ru.count_rows())
mt_ru.rows().select("allele_id", "svlen", "tr").show(10)

Rows with defined rule_applicable: 18354


+----------------+
| locus          |
+----------------+
| locus<GRCh38>  |
+----------------+
| chr22:25000196 |
| chr22:25000734 |
| chr22:25001794 |
| chr22:25003400 |
| chr22:25003400 |
| chr22:25003532 |
| chr22:25003664 |
| chr22:25004704 |
| chr22:25006273 |
| chr22:25006457 |
+----------------+

+------------------------------------------------------------------------------+
| alleles                                                                      |
+------------------------------------------------------------------------------+
| array<str>                                                                   |
+------------------------------------------------------------------------------+
| ["TTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGCGGGATCTCGGCT... |
| ["CTTGAACTCCTGGCCTCAAGTGAATCCGCCTGCCTCGGCCTCCGGAAG","C"]                     |
| ["A","ACGTGCCTGTTGGTCTTTCAGCTACAGAT"]                                        |
| ["A","ATTTTTTTTTTTTTTTTTTTT"]                                                |
| ["A","ATTTTTTTTTTTTTTTTTTTTTT"]                                              |
| ["C","CTCCTGAGCAGCTGGGACTACA"]                                               |
| ["GGCCTCCCAAAGTGCTGGGATTACAGGCGTGAACCACTGCGCCCGGCCCACTTGGAGATAATTTTTGTAAA... |
| ["T","TATTTTTTTTTTTTTTTTTTTTGTTTAAACCAA"]                                    |
| ["TCCACCCACCTCGGCTCCCAAGGTGTTGGGATTACAGATGTGAG","T"]                         |
| ["GTGTCTGGTCGCCTGGGGGGCGAC","G"]                                             |
+------------------------------------------------------------------------------+

+--------------------------+-------+-------------------+--------------------+
| allele_id                | svlen | tr.schema_version | tr.ruleset_version |
+--------------------------+-------+-------------------+--------------------+
| str                      | int32 | str               | str                |
+--------------------------+-------+-------------------+--------------------+
| "chr22-25000197-DEL-296" |   296 | "1"               | "1"                |
| "chr22-25000735-DEL-47"  |    47 | "1"               | "1"                |
| "chr22-25001795-INS-28"  |    28 | "1"               | "1"                |
| "chr22-25003401-INS-20"  |    20 | "1"               | "1"                |
| "chr22-25003401-INS-22"  |    22 | "1"               | "1"                |
| "chr22-25003533-INS-21"  |    21 | "1"               | "1"                |
| "chr22-25003665-DEL-405" |   405 | "1"               | "1"                |
| "chr22-25004705-INS-32"  |    32 | "1"               | "1"                |
| "chr22-25006274-DEL-43"  |    43 | "1"               | "1"                |
| "chr22-25006458-DEL-23"  |    23 | "1"               | "1"                |
+--------------------------+-------+-------------------+--------------------+

+-----------+--------------+------------+
| tr.contig | tr.pos_start | tr.pos_end |
+-----------+--------------+------------+
| str       |        int32 |      int32 |
+-----------+--------------+------------+
| "chr22"   |     25000195 |   25000492 |
| "chr22"   |     25000733 |   25000781 |
| "chr22"   |     25001793 |   25001822 |
| "chr22"   |     25003399 |   25003420 |
| "chr22"   |     25003399 |   25003422 |
| "chr22"   |     25003531 |   25003553 |
| "chr22"   |     25003663 |   25004069 |
| "chr22"   |     25004703 |   25004736 |
| "chr22"   |     25006272 |   25006316 |
| "chr22"   |     25006456 |   25006480 |
+-----------+--------------+------------+

+------------------------------------------------------------------------------+
| tr.ref                                                                       |
+------------------------------------------------------------------------------+
| str                                                                          |
+------------------------------------------------------------------------------+
| "TTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGCGGGATCTCG

In [9]:
mt_ru = mt1.filter_rows(
    hl.is_defined(mt1.tr.repeat_units_estimate)
    & hl.is_finite(mt1.tr.repeat_units_estimate)
)

print("Rows with defined repeat_units_estimate:", mt_ru.count_rows())
mt_ru.show(10)

Rows with defined repeat_units_estimate: 13333


,
locus,alleles
locus<GRCh38>,array<str>
chr22:25003400,"[""A"",""ATTTTTTTTTTTTTTTTTTTT""]"
chr22:25003400,"[""A"",""ATTTTTTTTTTTTTTTTTTTTTT""]"
chr22:25019050,"[""TTGTGTGTGTGTGTGTGTGTGTG"",""T""]"
chr22:25019050,"[""TTGTGTGTGTGTGTGTGTGTGTGTG"",""T""]"
chr22:25025088,"[""T"",""TAGCTACCAAGTGAAGCACATCACTTGGTAGC""]"
chr22:25025088,"[""T"",""TAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGC""]"
chr22:25025088,"[""T"",""TAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGC""]"
chr22:25025088,"[""T"",""TAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGCAGCTACCAAGTGAAGCACATCACTTGGTAGC""]"


## chr22 annotation summary stats

Quick sanity-check summaries for the main TR annotation categories on `mt1`.

In [10]:
# Set True only when you want extra grouped outputs (slower).
RUN_DETAILED_BREAKDOWNS = False

core = storm.print_tr_annotation_summary(
    mt1,
    run_detailed_breakdowns=RUN_DETAILED_BREAKDOWNS,
)

# `core` is a dict-like summary if you need values programmatically.
# print(core)

Core counts (single-pass aggregate)
rows_total: 18354
rows_with_tr_struct: 18354
rows_with_catalog_locus: 0
rows_match_method_best_overlap: 0
rows_match_method_no_overlap: 18354
rows_with_motif: 13333
rows_with_motif_matches_catalog_true: 0
rows_with_motif_matches_catalog_false: 0
rows_with_repeat_units_estimate: 13333
rows_rule_applicable_true: 0
rows_rule_applicable_false: 18354

Selected percentages
catalog_locus_pct: 0.000%
repeat_units_estimate_pct: 72.644%
rule_applicable_true_pct: 0.000%


## SAIGE prep (phase 1): feature strata inventory

This is the first implementation step for SAIGE integration. It builds a per-feature inventory from `mt1` and assigns each variant to one of the two primary testing strata:

- `standard_sv`
- `tr_quantitative`

In [11]:
feature_inventory = storm.build_feature_inventory(mt1)
storm.print_feature_inventory_stats(feature_inventory)

# Optional: persist a small feature manifest for SAIGE planning/debugging.
# feature_inventory.export("scratch/saige_feature_inventory.tsv.bgz")

Feature strata counts


,
feature_class,n
str,int64
"""tr_quantitative""",13333
"""standard_sv""",5021


TR quantitative features: catalog metadata snapshot


,,
has_catalog_locus,rule_applicable,n
bool,bool,int64
False,False,13333


## SAIGE prep (phase 2): project TR annotations to sample-level predictors

This step takes split-allele row annotations and projects them into entry/sample space using genotype dosage.

- `standard_sv` predictor: `dosage`
- `tr_quantitative` predictor: `dosage * repeat_units_estimate`

The resulting tables are long-format staging exports (`sample_id`, `feature_id`, `predictor`) that we can convert into SAIGE marker files in the next step.

In [12]:
tables = storm.build_long_predictor_tables(mt1)
standard_long = tables["standard_long"]
tr_quant_long = tables["tr_quant_long"]
storm.print_long_predictor_stats(standard_long, tr_quant_long)

# Optional exports for downstream SAIGE marker-file conversion.
# standard_long.export("scratch/saige_standard_long.tsv.bgz")
# tr_quant_long.export("scratch/saige_tr_quant_long.tsv.bgz")

Entry-level staging counts
standard_sv non-missing predictors: 63666280
tr_quantitative non-missing predictors: 169062440

Per-feature sample counts (tr_quantitative top 20)


,,
feature_id,n_samples,n_nonzero
str,int64,int64
"""chr22-27725556-INS-519""",12680,12649
"""chr22-31596957-DEL-45""",12680,12631
"""chr22-31903384-INS-428""",12680,12623
"""chr22-25820046-INS-1084""",12680,12550
"""chr22-25899800-INS-20""",12680,12324
"""chr22-33891014-DEL-28""",12680,12011
"""chr22-26860431-INS-24""",12680,11898
"""chr22-27805702-INS-110""",12680,11561


## SAIGE prep (phase 3): export staging artifacts

Export long predictor tables, feature-level QC summaries, and a sample manifest for SAIGE marker conversion.

In [13]:
# Faster export settings for local runs.
SAIGE_STAGE_DIR = REPO_ROOT / "scratch" / "saige_stage"
cols_ht = mt1.cols()
sample_manifest = cols_ht.select(sample_id=hl.str(cols_ht.s))
exports = storm.export_long_predictor_tables(
    standard_long,
    tr_quant_long,
    out_dir=SAIGE_STAGE_DIR,
    prefix="chr22",
    include_qc=False,  # optional and expensive
    include_sample_manifest=True,
    carriers_only=True,  # sparse export for speed
    output_format="ht",  # local backend: use Hail native tables for fast staging
    sample_manifest=sample_manifest,  # avoid expensive union/distinct on long tables
)

print("Exported SAIGE staging artifacts (fast mode):")
for name, path in exports.items():
    print(f"- {name}: {path}")

Exported SAIGE staging artifacts (fast mode):
- standard_long: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.standard_long.ht
- tr_quant_long: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.tr_quant_long.ht
- sample_manifest: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.sample_manifest.tsv
- carriers_only: True
- output_format: ht


## SAIGE prep: Step 2 dosage VCFs (dense)

`storm.export_saige_stratum_vcfs` writes two **`.vcf.bgz`** files (FORMAT **`DS`** only) for `standard_sv` and `tr_quantitative`, with **the same sample order** as the manifest derived from `mt1.cols()`. See `docs/saige_phewas_plan.md` (Phase 2).

- **SAIGE Step 2:** pass `--vcfField=DS` together with `--vcfFile` and `--vcfFileIndex`. If you did not use `tabix=True` in Hail, index with e.g. `tabix --csi -p vcf <path>.vcf.bgz`.
- **Scope:** this uses the **in-memory** `mt1` dense path (explicit zeros for non-carriers per stratum). It does **not** read the sparse `*.standard_long.ht` / `*.tr_quant_long.ht` staging tables; for replay from those, use `storm.build_feature_vcf_row_lookup` + `storm.dense_marker_matrix_from_long` in a separate script.
- **Cost:** full chr22 can be large and slow; the code cell below defaults to **skip** until you set `EXPORT_SAIGE_STEP2_VCF = True`.


In [20]:
# Dense Step-2 VCFs from mt1 (SAIGE --vcfField=DS). Can be slow / large for full chr22.
EXPORT_SAIGE_STEP2_VCF = True

cols_ht = mt1.cols()
manifest_for_step2 = cols_ht.select(sample_id=hl.str(cols_ht.s)).order_by("sample_id")

if EXPORT_SAIGE_STEP2_VCF:
    vcf_exports = storm.export_saige_stratum_vcfs(
        mt1,
        manifest_for_step2,
        out_dir=SAIGE_STAGE_DIR,
        prefix="chr22",
        tabix=False,
    )
    print("SAIGE Step 2 dosage VCFs (paths also under exports if you log them):")
    for k, p in sorted(vcf_exports.items()):
        print(f"  {k}: {p}")
    print("\nNext: index each VCF if needed, then SAIGE step2_SPAtests.R with --vcfField=DS.")
else:
    print("Skipping Step-2 VCF export (set EXPORT_SAIGE_STEP2_VCF = True).")


SAIGE Step 2 dosage VCFs (paths also under exports if you log them):
  standard_sv: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.standard_sv.ds.vcf.bgz
  tr_quantitative: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.tr_quantitative.ds.vcf.bgz

Next: index each VCF if needed, then SAIGE step2_SPAtests.R with --vcfField=DS.


In [22]:
!tabix --csi -f -p vcf /Users/kiran/repositories/storm/scratch/saige_stage/chr22.standard_sv.ds.vcf.bgz
!tabix --csi -f -p vcf /Users/kiran/repositories/storm/scratch/saige_stage/chr22.tr_quantitative.ds.vcf.bgz

## SAIGE prep (Phase 3 stub): synthetic phenotype + covariates

**Placeholder only** — deterministic toy `pheno_binary`, `pheno_quant`, `sex`, `age`, `seq_depth_log`, and `PC1`–`PC5` from each `sample_id` (Hail `hash`). Same IDs and order as `mt1` columns when you build the manifest below.

SAIGE Step 1: `--phenoFile` = exported TSV, `--sampleIDColinphenoFile=IID`, `--phenoCol=pheno_binary` (or `pheno_quant` for linear), `--covarColList` = printed string from `storm.saige_synthetic_covar_col_list()`.

Replace this file with real phenotypes and PCs before publishing results.


In [30]:
# Tab-delimited file for SAIGE step1_fitNULLGLMM.R (same cohort as marker VCFs).
try:
    _stage = SAIGE_STAGE_DIR
except NameError:
    _stage = REPO_ROOT / "scratch" / "saige_stage"
_stage.mkdir(parents=True, exist_ok=True)

cols_ht_pheno = mt1.cols()
pheno_manifest = cols_ht_pheno.select(sample_id=hl.str(cols_ht_pheno.s)).order_by("sample_id")
synthetic_pheno = storm.build_synthetic_saige_pheno_covar_table(pheno_manifest, n_pcs=5)
covar_list = storm.saige_synthetic_covar_col_list(n_pcs=5)

PHENO_COVAR_TSV = str(_stage / "chr22.saige_synthetic_pheno_covar.tsv")
storm.export_saige_phenotype_covariate_tsv(synthetic_pheno, PHENO_COVAR_TSV)

print("Synthetic phenotype + covariate TSV:", PHENO_COVAR_TSV)
print("Rows:", synthetic_pheno.count(), "| expected:", mt1.count_cols())
print("SAIGE --covarColList:", covar_list)
print("Example: --phenoCol=pheno_binary --sampleIDColinphenoFile=IID")


Synthetic phenotype + covariate TSV: /Users/kiran/repositories/storm/scratch/saige_stage/chr22.saige_synthetic_pheno_covar.tsv
Rows: 12680 | expected: 12680
SAIGE --covarColList: sex,age,seq_depth_log,PC1,PC2,PC3,PC4,PC5
Example: --phenoCol=pheno_binary --sampleIDColinphenoFile=IID


## SAIGE Step 1 + Step 2 via Pixi (`Rscript`)

The [SAIGE installation guide](https://saigegit.github.io/SAIGE-doc/docs/Installation.html) uses **Pixi**. The next **Python** cell runs `pixi run Rscript …/run_saige.R` with `subprocess`, sets **`STORM_SAIGE_RSCRIPT`** for nested Step 1/2, and prepends **`~/.pixi/bin`** to `PATH` so `pixi` is found even when Jupyter never sourced your `~/.zshrc`.

**Why not `%%bash`?** A bash magic subprocess often does **not** inherit variables from a prior `os.environ[...]` cell, and it may not see `pixi` on `PATH`. One Python cell avoids that.

**Before a full run:** edit `scripts/saige/example_chr22.config` and set **`plink_prefix`** and/or sparse GRM paths. Requires **`REPO_ROOT`** from earlier cells.


In [36]:
# Run Storm's SAIGE driver using Pixi (subprocess — reliable PATH + env in Jupyter).
import os
import shutil
import subprocess
from pathlib import Path

if "REPO_ROOT" not in dir():
    raise NameError("REPO_ROOT not defined — run the notebook cells that set the repo root first.")

repo = Path(REPO_ROOT).resolve()
# Directory that contains SAIGE's pixi.toml (edit if yours differs):
saige_src = Path.home() / "repositories" / "SAIGE"


def find_pixi_exe() -> str:
    w = shutil.which("pixi")
    if w:
        return w
    cand = Path.home() / ".pixi" / "bin" / "pixi"
    if cand.is_file():
        return str(cand)
    raise FileNotFoundError(
        "pixi not found (tried PATH and ~/.pixi/bin/pixi). Install Pixi or add it to PATH."
    )


pixi = find_pixi_exe()
if not (saige_src / "pixi.toml").is_file():
    raise FileNotFoundError(
        f"Expected pixi.toml under saige_src={saige_src}. Edit saige_src in this cell."
    )

env = os.environ.copy()
env["STORM_REPO"] = str(repo)
env["SAIGE_EXTDATA"] = str(saige_src / "extdata")
pixi_bin = str(Path(pixi).parent)
if pixi_bin not in env.get("PATH", ""):
    env["PATH"] = pixi_bin + os.pathsep + env.get("PATH", "")

rpath = subprocess.run(
    [pixi, "run", "bash", "-lc", "command -v Rscript"],
    cwd=str(saige_src),
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
if rpath.returncode != 0:
    raise RuntimeError(
        "Could not resolve Rscript inside pixi. stderr:\n" + (rpath.stderr or rpath.stdout)
    )
env["STORM_SAIGE_RSCRIPT"] = rpath.stdout.strip()
print("STORM_SAIGE_RSCRIPT =", env["STORM_SAIGE_RSCRIPT"])
subprocess.run([env["STORM_SAIGE_RSCRIPT"], "--version"], env=env, check=True)

driver = repo / "scripts" / "saige" / "run_saige.R"
cfg = repo / "scripts" / "saige" / "example_chr22.config"
cmd = [
    pixi,
    "run",
    "Rscript",
    str(driver),
    "--config",
    str(cfg),
    "--repo-root",
    str(repo),
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(saige_src), env=env)
if proc.returncode != 0:
    raise RuntimeError(f"SAIGE driver exited with code {proc.returncode}")


STORM_REPO = /Users/kiran/repositories/storm
SAIGE_SRC  = /Users/kiran/repositories/SAIGE


In [1]:
# Optional: same workflow in Terminal (after `cd` to SAIGE_SRC and fixing paths):
#   export PATH="$HOME/.pixi/bin:$PATH"
#   export SAIGE_EXTDATA="$PWD/extdata"
#   export STORM_SAIGE_RSCRIPT="$(pixi run bash -lc 'command -v Rscript')"
#   export STORM_REPO="/path/to/storm"
#   pixi run Rscript "$STORM_REPO/scripts/saige/run_saige.R" --config "$STORM_REPO/scripts/saige/example_chr22.config" --repo-root "$STORM_REPO"
print("(SAIGE is run from the Python cell above; this cell is reference only.)")


bash: line 3: STORM_REPO: Run the Python cell above first (sets STORM_REPO).


CalledProcessError: Command 'b'# Run Storm\'s SAIGE driver inside the Pixi environment (SAIGE docs: pixi run \xe2\x80\xa6).\nset -euo pipefail\n: "${STORM_REPO:?Run the Python cell above first (sets STORM_REPO).}"\n: "${SAIGE_SRC:?}"\n\nif [[ ! -f "$SAIGE_SRC/pixi.toml" ]]; then\n  echo "Expected pixi.toml under SAIGE_SRC=$SAIGE_SRC" >&2\n  exit 1\nfi\n\ncd "$SAIGE_SRC"\nexport SAIGE_EXTDATA="$SAIGE_SRC/extdata"\nexport STORM_SAIGE_RSCRIPT="$(pixi run bash -lc \'command -v Rscript\')"\n\necho "STORM_SAIGE_RSCRIPT=$STORM_SAIGE_RSCRIPT"\n"$STORM_SAIGE_RSCRIPT" --version\n\n# Uncomment for argument help only:\n# pixi run Rscript "$STORM_REPO/scripts/saige/run_saige.R" --help\n\npixi run Rscript "$STORM_REPO/scripts/saige/run_saige.R" \\\n  --config "$STORM_REPO/scripts/saige/example_chr22.config" \\\n  --repo-root "$STORM_REPO"\n'' returned non-zero exit status 1.

## SAIGE prep spot-checks

Quick integrity checks for exported stage-3 artifacts.

In [31]:
# Spot-check stage-3 outputs from the most recent export.
# Assumes prefix="chr22" and SAIGE_STAGE_DIR from the export cell.

std_ht_path = str(SAIGE_STAGE_DIR / "chr22.standard_long.ht")
tr_ht_path = str(SAIGE_STAGE_DIR / "chr22.tr_quant_long.ht")
manifest_path = str(SAIGE_STAGE_DIR / "chr22.sample_manifest.tsv")

std_ht = hl.read_table(std_ht_path)
tr_ht = hl.read_table(tr_ht_path)
manifest_ht = hl.import_table(manifest_path, delimiter="\t", impute=True).select("sample_id")

n_cols = mt1.count_cols()
manifest_n = manifest_ht.count()
manifest_unique_n = manifest_ht.aggregate(hl.len(hl.agg.collect_as_set(manifest_ht.sample_id)))

print("Sample manifest checks")
print("- mt1.count_cols():", n_cols)
print("- manifest rows:", manifest_n)
print("- manifest distinct sample_id:", manifest_unique_n)
print("- count matches mt1 cols:", manifest_n == n_cols)
print("- no duplicate sample_id:", manifest_n == manifest_unique_n)

std_n = std_ht.count()
std_zero = std_ht.aggregate(hl.agg.count_where(std_ht.predictor == 0))
tr_n = tr_ht.count()
tr_zero = tr_ht.aggregate(hl.agg.count_where(tr_ht.predictor == 0))

print("\nSparse predictor checks (carriers_only=True expected)")
print("- standard_long rows:", std_n)
print("- standard_long zero predictors:", std_zero)
print("- tr_quant_long rows:", tr_n)
print("- tr_quant_long zero predictors:", tr_zero)
print("- standard_long sparse OK:", std_zero == 0)
print("- tr_quant_long sparse OK:", tr_zero == 0)

print("\nTop feature support (tr_quant_long)")
tr_ht.group_by(feature_id=tr_ht.feature_id).aggregate(
    n_samples=hl.agg.count(),
    n_nonzero=hl.agg.count_where(tr_ht.predictor != 0),
).order_by(hl.desc("n_nonzero"), hl.desc("n_samples")).show(10)

Sample manifest checks
- mt1.count_cols(): 12680
- manifest rows: 12680
- manifest distinct sample_id: 12680
- count matches mt1 cols: True
- no duplicate sample_id: True

Sparse predictor checks (carriers_only=True expected)
- standard_long rows: 598286
- standard_long zero predictors: 0
- tr_quant_long rows: 2097120
- tr_quant_long zero predictors: 0
- standard_long sparse OK: True
- tr_quant_long sparse OK: True

Top feature support (tr_quant_long)


,,
feature_id,n_samples,n_nonzero
str,int64,int64
"""chr22-27725556-INS-519""",12649,12649
"""chr22-31596957-DEL-45""",12631,12631
"""chr22-31903384-INS-428""",12623,12623
"""chr22-25820046-INS-1084""",12550,12550
"""chr22-25899800-INS-20""",12324,12324
"""chr22-33891014-DEL-28""",12011,12011
"""chr22-26860431-INS-24""",11898,11898
"""chr22-27805702-INS-110""",11561,11561


## Cleanup

Call when finished (especially if re-running `init` in the same kernel).

In [ ]:
# hl.stop()